# RGB to grayscale conversion using CUDA
Done by: Siddharth Sudhakar (25901335)

In [1]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 11.7 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659448 sha256=ed1965bf43d0d98ae3073536005e0b407c86c997a8e92f6e9a84a8c523070a8c
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [2]:
import numpy as np
import time
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

In [20]:
from google.colab import files

In [21]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
img = Image.open(filename).convert("RGB").resize((4096, 4096))
img_array = np.array(img, dtype=np.float32)
print(f"Image shape: {img_array.shape}")

Saving RR.jpeg to RR (1).jpeg
Image shape: (4096, 4096, 3)


## CPU Implementation

In [4]:
def rgb_to_gray_cpu(img):
    return (0.2989 * img[:, :, 0] +
            0.5870 * img[:, :, 1] +
            0.1140 * img[:, :, 2]).astype(np.float32)

In [5]:
N_RUNS = 10
cpu_times = []
for _ in range(N_RUNS):
    start = time.perf_counter()
    gray_cpu = rgb_to_gray_cpu(img_array)
    cpu_times.append(time.perf_counter() - start)

In [6]:
cpu_mean = np.mean(cpu_times) * 1000
cpu_std  = np.std(cpu_times)  * 1000
print(f"CPU Mean: {cpu_mean:.2f} ms | Std: {cpu_std:.2f} ms")

CPU Mean: 130.26 ms | Std: 2.91 ms


## GPU Implementation

In [7]:
import pycuda.autoinit
import pycuda.driver as drv
from pycuda.compiler import SourceModule

In [8]:
cuda_kernel = """
__global__ void rgb_to_gray(
    const float* __restrict__ src,
    float*       __restrict__ dst,
    int width, int height)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;  // column
    int y = blockIdx.y * blockDim.y + threadIdx.y;  // row

    if (x < width && y < height) {
        int idx      = (y * width + x) * 3;
        float r      = src[idx    ];
        float g      = src[idx + 1];
        float b      = src[idx + 2];
        dst[y * width + x] = 0.2989f * r + 0.5870f * g + 0.1140f * b;
    }
}
"""

In [9]:
mod    = SourceModule(cuda_kernel)
kernel = mod.get_function("rgb_to_gray")

In [10]:
H, W = img_array.shape[:2]
img_flat = img_array.reshape(-1).copy()

In [11]:
src_gpu = drv.mem_alloc(img_flat.nbytes)
dst_gpu = drv.mem_alloc(H * W * 4)

In [12]:
BLOCK = (16, 16, 1)
GRID  = ((W + 15) // 16, (H + 15) // 16, 1)

In [13]:
gpu_times = []

In [15]:
for _ in range(N_RUNS):
    start = time.perf_counter()
    drv.memcpy_htod(src_gpu, img_flat)
    kernel(src_gpu, dst_gpu,
           np.int32(W), np.int32(H),
           block=BLOCK, grid=GRID)
    drv.Context.synchronize()
    gray_gpu = np.empty(H * W, dtype=np.float32)
    drv.memcpy_dtoh(gray_gpu, dst_gpu)
    gpu_times.append(time.perf_counter() - start)

In [16]:
gray_gpu = gray_gpu.reshape(H, W)
gpu_mean = np.mean(gpu_times) * 1000
gpu_std  = np.std(gpu_times)  * 1000

In [18]:
print(f"GPU | Mean: {gpu_mean:.2f} ms | Std: {gpu_std:.2f} ms")

GPU | Mean: 67.27 ms | Std: 2.08 ms


In [19]:
print(f"\nSpeedup: {cpu_mean / gpu_mean:.2f}x")


Speedup: 1.94x
